In [1]:
from random import random
from functools import reduce
from collections import namedtuple
from queue import PriorityQueue, SimpleQueue, LifoQueue
import numpy as np

In [5]:
PROBLEM_SIZE = 5  # number of elements in a set
NUM_SETS = 10
# Creates an array of arrays where one element has a 30% chance of being true. A set indicates which item is inside the set and which is not, if an element of the set has a value of true it means that it is present otherwise not
SETS = tuple(np.array([random() < .3 for _ in range(PROBLEM_SIZE)]) for _ in range(NUM_SETS))
State = namedtuple('State', ['taken', 'not_taken'])

# a state is made up of: the sets of items that I take and the sets that I don't take
# -> state=({1,3,5}, {0,2,4,6,7}) I then take the second array of items from SETS (the second set), the fourth and the sixth (take into account the array index)

In [3]:
# The goal is that all items must be taken with a certain number of state sets.
# The quality of a solution (the chosen state) is given by the smallest number of sets taken to get all the elements
# We need to do an or between all the elements of all the sets of the state to see if a certain element is present among all the sets or not (true is present, false is not)
# With all, we check if all the elements are present, that is, if the reduce gives me an array of True which means that all the elements are present among the sets taken
def goal_check(state):
    return np.all(reduce(np.logical_or, [SETS[i] for i in state.taken], np.array([False for _ in range(PROBLEM_SIZE)])))


# Calculate the distance from the input state to the final goal (for a greedy search), in short, it returns how many elements that need to be taken are missing
def distance(state):
    return PROBLEM_SIZE - sum(
        reduce(np.logical_or, [SETS[i] for i in state.taken], np.array([False for _ in range(PROBLEM_SIZE)])))

In [6]:
assert goal_check(State(set(range(NUM_SETS)), set())), "Problem not solvable"

In [ ]:
# Implementation using breadth-first search algorithm using a Simple Queue which would be a FIFO
# If we used a LifoQueue it would become a depth-first search

frontier = SimpleQueue()
frontier.put(State(set(), set(range(NUM_SETS))))

counter = 0
current_state = frontier.get()
while not goal_check(current_state):
    counter += 1
    for action in current_state[1]:
        new_state = State(current_state.taken ^ {action},
                          current_state.not_taken ^ {action})  # {1,2,3} ^ {2} = {1,3}  {1,3} ^ {2} = {1,2,3}
        frontier.put(new_state)
    current_state = frontier.get()

print(f"Solved in {counter:,} steps")
current_state

In [10]:
# Search algorithm implementation using Dijkstra
# The cost used for the PriorQueue is the number of elements in a set i.e., those taken

frontier = PriorityQueue()
frontier.put((0, (State(set(), set(range(NUM_SETS))))))

counter = 0
current_state = frontier.get()
while not goal_check(current_state[1]):
    counter += 1
    for action in current_state[1][1]:
        new_state = State(current_state[1].taken ^ {action},
                           current_state[1].not_taken ^ {action})  # {1,2,3} ^ {2} = {1,3}  {1,3} ^ {2} = {1,2,3}
        frontier.put((len(new_state.taken),new_state))
    current_state = frontier.get()

print(f"Solved in {counter:,} steps")
current_state

Solved in 39 steps


(2, State(taken={1, 2}, not_taken={0, 3, 4, 5, 6, 7, 8, 9}))

In [ ]:
# greedy search algorithm implementation

# frontier = PriorityQueue()
frontier = SimpleQueue()
state = State(set(), set(range(NUM_SETS)))
frontier.put((distance(state), state))

counter = 0
_, current_state = frontier.get()
while not goal_check(current_state):
    counter += 1
    for action in current_state[1]:
        new_state = State(current_state.taken ^ {action},
                          current_state.not_taken ^ {action})  # {1,2,3} ^ {2} = {1,3}  {1,3} ^ {2} = {1,2,3}
        frontier.put((distance(new_state), new_state))
    _, current_state = frontier.get()

print(f"Solved in {counter:,} steps")
current_state